## Importing libraries
<br>
numpy, pandas, scikit learn and tenserflow


In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Importing the Dataset

In [5]:
data = pd.read_csv("news.csv")
data.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [6]:
data.head(5)

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [7]:
data.tail(5)

,Unnamed: 0,title,text,label
6330,4490,State Department says it can't find emails fro...,The State Department told the Republican Natio...,REAL
6331,8062,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,FAKE
6332,8622,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,FAKE
6333,4021,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene...",REAL
6334,4330,Jeb Bush Is Suddenly Attacking Trump. Here's W...,Jeb Bush Is Suddenly Attacking Trump. Here's W...,REAL


#processing Dataset

In [8]:
data = data.drop(["Unnamed: 0"], axis=1)
data.head(5)

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


Now the data is cleaned we can go for data encoding.

#Data Encoding
<br>
Transforms the categorical labels into numerical format....
<br> (0 for REAL, 1 for FAKE)

In [9]:
le = preprocessing.LabelEncoder()
le.fit(data['label'])
data['label'] = le.transform(data['label'])

In [10]:
data.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",0
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,0
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,1
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",0
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,1


# Variables Setup

In [20]:
embedding_dim = 50
max_length = 54
padding_type = 'post'
trunc_type = 'post'
oov_tok = "<OOV>"
training_size = 3000
test_portion = 0.1

# Tokenization
<br> This process divides a large piece of continuous text into distinct units or tokens. Here we use columns separately for a temporal basis as a pipeline just for good accuracy.

tokenizer1.fit_on_texts(title): Fits the tokenizer on the 'title' column to create a vocabulary. <br>
pad_sequences(sequences1): Pads the sequences to ensure they all have the same length.


In [12]:
title = []
text = []
labels = []
for x in range(training_size):
  title.append(data['title'][x])
  text.append(data['text'][x])
  labels.append(data['label'][x])
tokenizer1 = Tokenizer()
tokenizer1.fit_on_texts(title)
word_index1 = tokenizer1.word_index
vocab_size1 = len(word_index1)
sequences1 = tokenizer1.texts_to_sequences(title)
padded1 = pad_sequences(sequences1, padding=padding_type,
                        truncating=trunc_type)

this is ready for deep learning mode

# Splitting Data for Training and Testing
<br>
training_sequences1, test_sequences1: Splits the tokenized and padded data into training and testing sets.<br>
training_labels, test_labels: Splits the corresponding labels into training and testing labels.

In [13]:
split = int(test_portion * training_size)
training_sequences1 = padded1[split:training_size]
test_sequences1 = padded1[0:split]
test_labels = labels[0:split]
training_labels = labels[split:training_size]

# Reshaping Data for LSTM
<br> We will be using LSTM(Long Short Term Memory) model for prediction and for that we need to reshape padded sequence. We are converting it into np.array() as we need training and test sequences into NumPy arrays which are required by TensorFlow models.

In [14]:
training_sequences1 = np.array(training_sequences1)
test_sequences1 = np.array(test_sequences1)

# Generating Word Embedding <br>
Embeddings allows words with similar meanings to have a similar representation. here each individual word is represented as real-valued vectors in a predefined vector space. For that we will be using glove.6B.50d.txt.

In [15]:
!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!unzip glove.6B.zip

--2026-02-15 12:27:03--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.01MB/s    in 2m 39s  

2026-02-15 12:29:43 (5.16 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflating: glove.6B.200d.txt       
  inflating: glove.6B.300d.txt       


now that our glove embeddings are downloaded we can use them for word embedding.. <br> Text → Tokenizer → Convert to sequences <br>
<br>
         ↓
   <br>
Load GloVe embeddings <br>
         ↓
   <br>
Create embedding matrix <br>
         ↓
   <br>
Use in Embedding layer <br>
         ↓
   <br>
Train LSTM/CNN model


In [16]:
embedding_index = {}
with open('glove.6B.50d.txt', 'r', encoding='utf-8') as f:
  for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embedding_index[word] = coefs

embedding_matrix = np.zeros((vocab_size1 + 1, embedding_dim))

for word, i in word_index1.items():
  if i < vocab_size1:
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
      embedding_matrix[i] = embedding_vector

Tokenizer gives word → index
<br>
GloVe gives word → vector
<br>
Embedding matrix connects both
<br>
Neural network now understands meaning of words
<br>

Download GloVe<br>
                 ↓
                 <br>
Load GloVe into dictionary<br>
                 ↓
                 <br>
Create empty embedding matrix<br>
                 ↓
                 <br>
Match tokenizer words with GloVe<br>
                 ↓
                 <br>
Fill embedding matrix<br>
                 ↓
                 <br>
Pass into Embedding layer


# Model Architecture
We use now TensorFlow embedding technique with Keras Embedding Layer. <br>
this layer uses pre-trained GloVe embedding.

In [21]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size1 + 1, embedding_dim, input_length=max_length,
                              weights=[embedding_matrix], trainable=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Conv1D(64, 5, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=4),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │       377,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 377,600 (1.44 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 377,600 (1.44 MB)

Now that our model architecture is ready we can use this to train our model

# Training the Model

In [23]:
history = model.fit(
    training_sequences1,
    np.array(training_labels),
    epochs=50,
    validation_data=(test_sequences1, np.array(test_labels)),
    verbose=2
)

Epoch 1/50
85/85 - 6s - 67ms/step - accuracy: 0.6196 - loss: 0.6359 - val_accuracy: 0.6967 - val_loss: 0.5666
Epoch 2/50
85/85 - 1s - 15ms/step - accuracy: 0.7107 - loss: 0.5618 - val_accuracy: 0.7200 - val_loss: 0.5259
Epoch 3/50
85/85 - 1s - 14ms/step - accuracy: 0.7352 - loss: 0.5233 - val_accuracy: 0.7100 - val_loss: 0.5205
Epoch 4/50
85/85 - 1s - 14ms/step - accuracy: 0.7630 - loss: 0.4779 - val_accuracy: 0.7500 - val_loss: 0.5080
Epoch 5/50
85/85 - 1s - 14ms/step - accuracy: 0.8141 - loss: 0.4109 - val_accuracy: 0.7367 - val_loss: 0.5127
Epoch 6/50
85/85 - 2s - 19ms/step - accuracy: 0.8270 - loss: 0.3816 - val_accuracy: 0.6833 - val_loss: 0.6160
Epoch 7/50
85/85 - 3s - 38ms/step - accuracy: 0.8356 - loss: 0.3658 - val_accuracy: 0.7367 - val_loss: 0.5543
Epoch 8/50
85/85 - 2s - 18ms/step - accuracy: 0.8707 - loss: 0.3113 - val_accuracy: 0.7833 - val_loss: 0.4967
Epoch 9/50
85/85 - 1s - 14ms/step - accuracy: 0.8881 - loss: 0.2740 - val_accuracy: 0.7533 - val_loss: 0.5356
Epoch 10/5

For each epoch training accuracy improves reaching around 97% by the 50th epoch while the validation accuracy is around 75%. The validation loss gradually decreases, indicating that the model is learning from the data but it also shows signs of some overfitting as the validation accuracy is lower than the training accuracy. To avoid this we can further fine tune the model


# Sample Prediction

In [28]:
X = "Karry to go to France in gesture of sympathy"

sequences = tokenizer1.texts_to_sequences([X])
sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)
if model.predict(sequences, verbose=0)[0][0] >= 0.5:
    print("This news is True")
else:
    print("This news is False")


This news is False


As we can see our model is working fine and now can be used to detect of any information is fake or not.
<br>
By following these steps we successfully built a fake news detection model using TensorFlow in Python